# 「コラボ」のデータから見るにじさんじとホロライブの違い

対象期間: 2025-06-01から2026-05-31まで（Asia/Tokyo）。

このNotebookは、記事用の集計と図をBigQuery中心で再現するためのものです。YouTube Data APIで取得した動画メタデータをBigQueryへロードし、概要欄に出てくる既知チャンネルID / @handleをもとにコラボ候補edgeを作っています。BigQuery Graphの可視化は `%%bigquery --graph` を使います。

注意: これは「実コラボの完全な正解データ」ではなく、概要欄リンクに基づく候補グラフです。

In [ ]:
PROJECT = "jackojacko05"
DATASET = "nijiholo"
LOCATION = "asia-northeast1"
START_DATE = "2025-06-01"
END_DATE = "2026-05-31"

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT, location=LOCATION)

def q(sql: str):
    return client.query(sql, location=LOCATION).to_dataframe()


## 1. データの全体像

記事ではまず、対象期間・動画行数・コラボ候補行数を明示します。後段の主張が期間指定のデータに基づくことをここで固定します。

In [ ]:
scope = q("""
SELECT *
FROM `jackojacko05.nijiholo.article_scope`
""")
scope

## 2. 供給と需要を分ける

この記事で一番大事なのは、コラボの「多さ」と、コラボ時の「上振れ」を分けることです。今回のデータでは、供給量はホロライブも高く、にじさんじが一方的に多いとは言いにくい。一方で、同一ライバー内の相対再生数では、にじさんじのコラボ候補動画の上振れが大きく出ます。

In [ ]:
supply = q("""
SELECT
  org,
  talents,
  owner_video_rows,
  collab_owner_video_rows,
  collab_share,
  cross_org_share
FROM `jackojacko05.nijiholo.article_org_supply`
ORDER BY org
""")

owner_uplift = q("""
SELECT
  org,
  comparable_talents,
  median_collab_view_uplift_pct,
  median_collab_view_per_day_uplift_pct,
  positive_view_uplift_talents
FROM `jackojacko05.nijiholo.article_owner_uplift`
ORDER BY org
""")

display(supply)
display(owner_uplift)

In [ ]:
import matplotlib.pyplot as plt

org_label = {"nijisanji": "にじさんじ", "hololive": "ホロライブ"}
colors = {"nijisanji": "#2563eb", "hololive": "#06b6d4"}

merged = supply.merge(owner_uplift, on="org")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

axes[0].barh([org_label[o] for o in merged["org"]], merged["collab_share"] * 100, color=[colors[o] for o in merged["org"]])
axes[0].set_title("コラボ候補あり動画率")
axes[0].set_xlabel("%")

axes[1].barh([org_label[o] for o in merged["org"]], merged["median_collab_view_uplift_pct"] * 100, color=[colors[o] for o in merged["org"]])
axes[1].set_title("同一ライバー内のコラボ上振れ中央値")
axes[1].set_xlabel("%")

fig.suptitle("コラボは供給より上振れに差が出た")
fig.tight_layout()
fig.savefig("../reports/note_figures/notebook_fig_supply_demand.png", dpi=180)
fig

## 3. ジャンル別に分解する

ジャンルはタイトルのルールベース分類です。歌系を先に判定し、その後ゲーム系、それ以外をその他にしています。

ゲームは両社ともコラボ率が高く、差はあまり本丸ではありません。差が見えやすいのは、歌とその他です。

In [ ]:
genre_summary = q("""
SELECT *
FROM `jackojacko05.nijiholo.article_genre_summary`
ORDER BY CASE genre WHEN 'ゲーム' THEN 1 WHEN '歌' THEN 2 ELSE 3 END, org
""")

genre_uplift = q("""
SELECT *
FROM `jackojacko05.nijiholo.article_genre_uplift`
ORDER BY CASE genre WHEN 'ゲーム' THEN 1 WHEN '歌' THEN 2 ELSE 3 END, org
""")

display(genre_summary)
display(genre_uplift)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.8))
genres = ["ゲーム", "歌", "その他"]
y_base = range(len(genres))
height = 0.34

for offset, org in [(-height / 2, "nijisanji"), (height / 2, "hololive")]:
    values = []
    for genre in genres:
        row = genre_uplift[(genre_uplift["org"] == org) & (genre_uplift["genre"] == genre)].iloc[0]
        values.append(row["median_collab_view_uplift_pct"] * 100)
    ax.barh([y + offset for y in y_base], values, height=height, label=org_label[org], color=colors[org])

ax.axvline(0, color="#94a3b8", linewidth=1)
ax.set_yticks(list(y_base), genres)
ax.set_xlabel("コラボ上振れ中央値（%）")
ax.set_title("ジャンル別: コラボ上振れは歌・その他で大きい")
ax.legend()
fig.tight_layout()
fig.savefig("../reports/note_figures/notebook_fig_genre_uplift.png", dpi=180)
fig

## 4. BigQuery Graphとして可視化する

裏テーマは、BigQuery Graphのサンプルです。`youtube_collab_graph` は `Character` node と `MentionsChannel` edgeで構成しています。

BigQuery Notebook / Colab Enterprise上では、BigQuery Graph可視化の形式に合わせて `%%bigquery --graph` でGQLを実行します。返り値は `TO_JSON(p) AS path` にします。Notebook可視化は返却データ量2MBの上限があるため、表示対象は上位edgeに絞ります。

In [ ]:
# BigQuery Graph visualization magic. Run once in the notebook runtime.
!pip install bigquery_magics==0.12.1
%load_ext bigquery_magics

import bigquery_magics
bigquery_magics.context.project = PROJECT
bigquery_magics.context.default_query_job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=100 * 1024 * 1024
)


### コラボ候補グラフ: 強いedgeを俯瞰する

まずは `video_count >= 8` のedgeに絞ります。にじさんじ内・ホロライブ内・横断の関係が混ざるため、全体の密度感を見る入口になります。

In [ ]:
%%bigquery --project jackojacko05 --graph
GRAPH `jackojacko05.nijiholo.youtube_collab_graph`
MATCH p = (owner:Character)-[mention:MentionsChannel]->(collaborator:Character)
WHERE mention.video_count >= 8
RETURN TO_JSON(p) AS path
ORDER BY mention.video_count DESC
LIMIT 120


### にじホロ横断edgeだけを見る

箱内の循環と横断コラボは意味が違うので、組織が異なるedgeだけを別に表示します。横断edgeは本数が少ないため、1本だけの候補も含めて確認します。

In [ ]:
%%bigquery --project jackojacko05 --graph
GRAPH `jackojacko05.nijiholo.youtube_collab_graph`
MATCH p = (owner:Character)-[mention:MentionsChannel]->(collaborator:Character)
WHERE owner.org != collaborator.org
RETURN TO_JSON(p) AS path
ORDER BY mention.video_count DESC
LIMIT 80


### 表でも確認する

記事の数値確認用に、同じgraphから上位edgeを表でも取り出します。

In [ ]:
graph_sample = q("""
SELECT *
FROM GRAPH_TABLE(
  `jackojacko05.nijiholo.youtube_collab_graph`
  MATCH (a:Character)-[e:MentionsChannel]->(b:Character)
  RETURN
    a.character_name AS owner_name,
    a.org AS owner_org,
    b.character_name AS collaborator_name,
    b.org AS collaborator_org,
    e.video_count AS video_count,
    e.sample_video_url AS sample_video_url,
    e.sample_title AS sample_title
)
ORDER BY video_count DESC
LIMIT 20
""")
graph_sample

In [ ]:
edge_matrix = q("""
SELECT *
FROM `jackojacko05.nijiholo.article_edge_matrix`
ORDER BY src_org, dst_org
""")
edge_matrix

## 5. 記事での使い方

- 本文の主張は「コラボ供給」ではなく「コラボ需要の上振れ」に置く。
- `箱推し / 単推し` は結論ではなく比喩として使う。
- `にじさんじはプロセカ、ホロライブはアイマス` も同じく比喩。今回のデータが直接証明しているわけではない。
- 断定しすぎず、概要欄リンク由来の候補グラフであることを明記する。
- 再現用コードとして、SQLは `sql/005_youtube_collab_graph.sql` と `sql/008_article_collab_metrics.sql` を提示する。